# Datenanalyse mit SQL & Python - Tag 4: Explorative Datenanalyse & Reporting

**Donnerstag:** EDA, Data Cleaning und Reporting mit einem echten messy Job-Posting-Datensatz.  
**Business-Fokus:** Welche Faktoren hängen mit Data-Science-Gehältern zusammen?


## Lernziele

Nach Tag 4 kannst du:

- Explorative Datenanalyse (EDA) mit statistischen Kennzahlen, Verteilungen und Korrelationen durchführen
- typische Datenqualitätsprobleme in einem realistischeren Rohdatensatz erkennen
- Text-, Gehalts- und Standortspalten mit Pandas bereinigen
- SQL und Python kombinieren, um komplexere Analysefragen zu beantworten
- aussagekräftige Diagramme und kurze Reports erstellen
- aus einer EDA eine datenbasierte Business-Empfehlung ableiten


## Themen des Tages

1. Data Cleaning Recap: fehlende Werte, Typen, Duplikate, Textspalten
1. Job-Posting-Daten verstehen und bereinigen
1. Gehaltsspannen extrahieren und analysierbar machen
1. Explorative Datenanalyse: Kennzahlen, Verteilungen, Ausreißer, Korrelationen
1. Kombination von SQL und Python für komplexe Analysen
1. Reporting: Von Diagrammen zu Erkenntnissen
1. Mini-Projekt: Faktoren für Data-Science-Gehälter untersuchen


## Einrichtung & Daten laden

Wir nutzen `Uncleaned_DS_jobs.csv` aus dem öffentlichen Repository `eyowhite/Messy-dataset`.

Quelle: https://github.com/eyowhite/Messy-dataset


In [ ]:
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 100)

JOB_DATA_URL = 'https://raw.githubusercontent.com/eyowhite/Messy-dataset/main/Uncleaned_DS_jobs.csv'

jobs_raw = pd.read_csv(JOB_DATA_URL, engine='python', on_bad_lines='warn')
print('Rohdaten:', jobs_raw.shape)
jobs_raw.head()


## 1. Einstieg: Warum ist dieser Datensatz besser für Tag 4?

Der Shop-Datensatz aus den vorherigen Tagen ist klein und relativ sauber. Für Tag 4 brauchen wir mehr Reibung:

- Textspalten mit Zusatzinformationen
- Gehaltsspannen als Strings
- uneinheitliche Firmenangaben
- negative oder fehlende Werte als Platzhalter
- mehrere mögliche Analysepfade

Genau deshalb eignet sich ein unbereinigter Job-Posting-Datensatz gut für EDA & Reporting.


## 2. Schnell-Check

Bevor wir bereinigen, prüfen wir Struktur, Datentypen, fehlende Werte und Beispiele aus den wichtigsten Spalten.


In [ ]:
print('Form:', jobs_raw.shape)
print()
print('Spalten:')
print(jobs_raw.columns.tolist())
print()
print('Datentypen:')
print(jobs_raw.dtypes)
print()
print('Fehlende Werte:')
print(jobs_raw.isna().sum().sort_values(ascending=False).head(20))


In [ ]:
jobs_raw.describe(include='all').T.head(20)


## 3. Data Cleaning: Job-Posting-Daten bereinigen

Wir machen den Datensatz nicht perfekt. Ziel ist eine solide Analysebasis für Gehälter, Standorte, Branchen und Ratings.


In [ ]:
jobs = jobs_raw.copy()

# Spaltennamen vereinheitlichen
jobs.columns = (
    jobs.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

# Textspalten trimmen
text_cols = jobs.select_dtypes(include=['object']).columns
for col in text_cols:
    jobs[col] = jobs[col].astype('string').str.strip()

# Häufiger Platzhalter im Dataset: -1 als fehlender Wert
jobs = jobs.replace(-1, np.nan).replace('-1', pd.NA)

jobs.head()


### 3.1 Gehaltsspanne extrahieren

Die Spalte `salary_estimate` ist für Analysen zentral, aber sie ist Text. Wir extrahieren minimale, maximale und durchschnittliche Gehaltswerte.


In [ ]:
salary_clean = (
    jobs['salary_estimate']
    .astype('string')
    .str.replace(r'\(.*\)', '', regex=True)
    .str.replace('K', '', regex=False)
    .str.replace('$', '', regex=False)
    .str.strip()
)

salary_parts = salary_clean.str.extract(r'(?P<min_salary>\d+)\s*-\s*(?P<max_salary>\d+)')
jobs['min_salary_k'] = pd.to_numeric(salary_parts['min_salary'], errors='coerce')
jobs['max_salary_k'] = pd.to_numeric(salary_parts['max_salary'], errors='coerce')
jobs['avg_salary_k'] = jobs[['min_salary_k', 'max_salary_k']].mean(axis=1)

jobs[['salary_estimate', 'min_salary_k', 'max_salary_k', 'avg_salary_k']].head(10)


### 3.2 Firmenname, Standort und Alter der Firma

Wir entfernen Ratings aus Firmennamen und berechnen ein grobes Firmenalter.


In [ ]:
if 'company_name' in jobs.columns:
    jobs['company_clean'] = jobs['company_name'].astype('string').str.replace(r'\n.*', '', regex=True).str.strip()

if 'founded' in jobs.columns:
    jobs['company_age'] = 2026 - pd.to_numeric(jobs['founded'], errors='coerce')
    jobs.loc[jobs['company_age'] < 0, 'company_age'] = np.nan

if 'location' in jobs.columns:
    jobs['state'] = jobs['location'].astype('string').str.extract(r',\s*([A-Z]{2})$')

jobs[['company_clean', 'location', 'state', 'company_age']].head(10)


## 4. Univariate EDA: Gehaltsverteilung und Ratings

Jetzt analysieren wir einzelne Variablen: Gehalt, Rating und Firmenalter.


In [ ]:
jobs[['min_salary_k', 'max_salary_k', 'avg_salary_k', 'rating', 'company_age']].describe().T


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(jobs['avg_salary_k'].dropna(), bins=25)
plt.title('Verteilung der geschätzten Durchschnittsgehälter')
plt.xlabel('Durchschnittsgehalt in Tsd. USD')
plt.ylabel('Anzahl Job Postings')
plt.show()


In [ ]:
plt.figure(figsize=(8, 3))
sns.boxplot(data=jobs, x='avg_salary_k')
plt.title('Ausreißercheck: Durchschnittsgehalt')
plt.xlabel('Durchschnittsgehalt in Tsd. USD')
plt.show()


## 5. Bivariate EDA: Gruppen vergleichen

Wir vergleichen Gehalt nach Standort, Branche oder Sektor. Je nach Datenqualität können einzelne Spalten leer sein.


In [ ]:
state_salary = (
    jobs
    .dropna(subset=['state', 'avg_salary_k'])
    .groupby('state')['avg_salary_k']
    .agg(['count', 'mean', 'median'])
    .query('count >= 5')
    .sort_values('mean', ascending=False)
)
state_salary.head(10)


In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(data=state_salary.head(10).reset_index(), x='mean', y='state')
plt.title('Top-Standorte nach durchschnittlichem Gehalt')
plt.xlabel('Durchschnittsgehalt in Tsd. USD')
plt.ylabel('State')
plt.show()


In [ ]:
if 'sector' in jobs.columns:
    sector_salary = (
        jobs
        .dropna(subset=['sector', 'avg_salary_k'])
        .groupby('sector')['avg_salary_k']
        .agg(['count', 'mean', 'median'])
        .query('count >= 5')
        .sort_values('mean', ascending=False)
    )
    display(sector_salary.head(10))


## 6. Korrelationen

Wir prüfen, ob numerische Variablen wie Rating, Firmenalter und Gehalt zusammenhängen.


In [ ]:
numeric_cols = [col for col in ['avg_salary_k', 'rating', 'company_age'] if col in jobs.columns]
corr = jobs[numeric_cols].corr()
corr


In [ ]:
sns.heatmap(corr, annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Korrelationen im Job-Posting-Datensatz')
plt.show()


## 7. SQL + Python für komplexere Analysen

Wir speichern den bereinigten DataFrame in SQLite und führen eine aggregierte Analyse aus.


In [ ]:
conn = sqlite3.connect(':memory:')
jobs.to_sql('jobs', conn, index=False, if_exists='replace')

query = '''
SELECT
    state,
    COUNT(*) AS postings,
    AVG(avg_salary_k) AS avg_salary_k,
    AVG(rating) AS avg_rating
FROM jobs
WHERE state IS NOT NULL
  AND avg_salary_k IS NOT NULL
GROUP BY state
HAVING COUNT(*) >= 5
ORDER BY avg_salary_k DESC;
'''

sql_state_report = pd.read_sql_query(query, conn)
sql_state_report.head(10)


## 8. Reporting: Von Diagrammen zu Erkenntnissen

Eine gute EDA endet mit einer kurzen, belastbaren Story. Beispielstruktur:

- Welche Gehaltsunterschiede sind sichtbar?
- Welche Standorte oder Sektoren fallen auf?
- Welche Datenqualitätsprobleme begrenzen die Aussage?
- Welche Empfehlung oder nächste Analyse folgt daraus?


In [ ]:
insights = [
    'Insight 1: Gehälter unterscheiden sich deutlich nach Standort.',
    'Insight 2: Nicht alle Felder sind sauber genug für jede Analysefrage.',
    'Insight 3: Für eine belastbare Empfehlung sollten Standort, Sektor und Job-Level gemeinsam betrachtet werden.'
]

for insight in insights:
    print('-', insight)


## Mini-Projekt: Welche Faktoren hängen mit Data-Science-Gehältern zusammen?

**Aufgabe:** Erstelle einen kurzen Report zur Frage:

> Welche Faktoren hängen im Job-Posting-Datensatz mit höheren Data-Science-Gehältern zusammen?

Mögliche Faktoren:

- Standort / State
- Sector oder Industry
- Rating
- Firmenalter
- Jobtitel oder Seniorität

**Ergebnis:** 2-3 Diagramme, 3 Erkenntnisse, 1 Empfehlung und 1 Hinweis zur Datenqualität.


## Zusammenfassung

Heute hast du EDA und Reporting an einem realistischeren Rohdatensatz geübt:

1. Rohdaten prüfen
1. zentrale Text- und Zahlenspalten bereinigen
1. Gehalt aus Text extrahieren
1. Verteilungen und Gruppen vergleichen
1. SQL und Python kombinieren
1. Erkenntnisse in eine Business-Story übersetzen
